# Step 3 — RAG Retrieval, Chatbot & Image Path

**Skincare Recommendation Chatbot** — this notebook builds the AI layer on top of the gold tables produced in Step 2:

1. **Hybrid RAG retrieval** — semantic embeddings (Chroma + sentence-transformers) combined with structured concern tags
2. **Grounded LLM chatbot** — Databricks-hosted Llama 3.1, constrained to recommend only real catalog products with their real ingredients
3. **Image path** — Llama 4 Maverick vision with three-mode safety logic: *recommend / escalate to dermatologist / retake*

**Inputs:** `gold_products`, `gold_product_concerns`, `concern_ingredient_kb` (Unity Catalog)
**Design iterations are documented in the markdown notes** — the code below is the final (best) version of each component.

## 1. Install tools

- `sentence-transformers` — turns text into embeddings (meaning vectors)
- `chromadb` — in-memory vector store for fast similarity search

*Note: this cell restarts Python — run it once, then run everything below.*

In [0]:
%pip install sentence-transformers chromadb databricks-sdk[openai] -q
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow-skinny 2.11.4 requires packaging<24, but you have packaging 26.2 which is incompatible.
mlflow-skinny 2.11.4 requires protobuf<5,>=3.12.0, but you have protobuf 5.29.3 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 2. Build the retrieval layer

One consolidated cell: load gold tables → build embedding documents → embed → load Chroma → concern detection.

**Design notes (iteration history):**
- *v1 documents* buried the concerns under 15 ingredients → retrieval was fuzzy (irrelevant products in top-5, distances ~1.08+).
- *Fix A:* documents rewritten to lead with concerns and trim ingredients to 5 → correct product rose to #1, distances dropped ~8%.
- *Fix B:* pure semantic search still leaked untagged products, so retrieval is **hybrid** — embeddings rank candidates, structured concern tags filter them. Concern detection maps free text ("greasy, breaking out") to knowledge-base concerns via cosine similarity, threshold 0.45 tuned so secondary concerns (e.g. "breakouts" → acne) are caught.

In [0]:
# ============================================================
# SETUP (a) — Load gold tables into pandas  [Spark step]
# ============================================================
# NOTE: if this cell throws ENVIRONMENT_SETUP_ERROR (a transient
# Databricks serverless issue after restartPython), just RE-RUN it.
from pyspark.sql import functions as F

gold_products = spark.table("dermstore.dermstoreschema.gold_products")
gold_concerns = spark.table("dermstore.dermstoreschema.gold_product_concerns")

concerns_per_product = gold_concerns.groupBy("product_id").agg(
    F.collect_list("concern").alias("concerns"))
products_full = gold_products.join(concerns_per_product, "product_id", "left")
pdf = products_full.toPandas()

all_concerns = [r["concern"] for r in
                spark.table("dermstore.dermstoreschema.concern_ingredient_kb")
                .select("concern").distinct().collect()]

print("Loaded", len(pdf), "products |", len(all_concerns), "concerns")

Loaded 106 products | 14 concerns


In [0]:
# ============================================================
# SETUP (b) — Build documents, embed, load Chroma  [no Spark]
# ============================================================
from sentence_transformers import SentenceTransformer
import chromadb
from numpy import dot
from numpy.linalg import norm

# --- Build one descriptive document per product (Fix A design) ---
def build_document(row):
    brand = row["brand"] or ""
    concerns = row["concerns"] if row["concerns"] is not None else []
    concerns_text = ", ".join(concerns) if len(concerns) else "general skincare"
    ingredients = row["ingredients_list"] if row["ingredients_list"] is not None else []
    key_ingredients = ", ".join(ingredients[:5])
    return (f"This skincare product helps with: {concerns_text}. "
            f"It is suitable for someone concerned about {concerns_text}. "
            f"Brand: {brand}. Contains: {key_ingredients}.")

pdf["document"] = pdf.apply(build_document, axis=1)

# --- Embed all products (384-dim vectors) ---
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(pdf["document"].tolist(), show_progress_bar=True)

# --- Load into Chroma ---
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("skincare_products")
except Exception:
    pass
collection = chroma_client.create_collection("skincare_products")
collection.add(
    ids=pdf["product_id"].astype(str).tolist(),
    embeddings=[e.tolist() for e in embeddings],
    documents=pdf["document"].tolist(),
    metadatas=[{"brand": str(r["brand"]),
                "price": float(r["price"]) if r["price"] is not None else 0.0,
                "concerns": ", ".join(r["concerns"]) if r["concerns"] is not None else ""}
               for _, r in pdf.iterrows()])

# --- Concern detection (free text -> knowledge-base concerns) ---
concern_embeddings = model.encode(all_concerns)

def detect_concerns(query_text, top_k=3, threshold=0.45):
    q_emb = model.encode([query_text])[0]
    scores = []
    for concern, c_emb in zip(all_concerns, concern_embeddings):
        cos = dot(q_emb, c_emb) / (norm(q_emb) * norm(c_emb))
        scores.append((concern, float(cos)))
    scores.sort(key=lambda x: -x[1])
    matched = [c for c, s in scores[:top_k] if s >= threshold]
    return matched if matched else [scores[0][0]]

# --- Ingredient lookup for grounding the LLM ---
id_to_ingredients = {
    str(row["product_id"]): (row["ingredients_list"][:8]
                             if row["ingredients_list"] is not None else [])
    for _, row in pdf.iterrows()}

print("Retrieval layer ready. Products in vector store:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Retrieval layer ready. Products in vector store: 106


## 3. LLM client (Databricks Foundation Model APIs)

Databricks notebooks ship a pre-authenticated OpenAI-compatible client — no API key needed. Text generation uses **Llama 3.1 8B** (pay-per-token endpoint, covered by Free Edition quota).

In [0]:
# LLM client — pre-authenticated inside Databricks
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

# Smoke test
resp = client.chat.completions.create(
    model="databricks-meta-llama-3-1-8b-instruct",
    messages=[{"role": "user", "content": "Say hello in one short sentence."}],
    max_tokens=50)
print(resp.choices[0].message.content)

Hello!


## 4. The chatbot (hybrid retrieval + grounded generation)

**Grounding design (v2 — final):** the first version hallucinated — it invented ingredient benefits and fake product URLs, and assumed concerns the user never mentioned. Fixes:
1. Each retrieved product's **real ingredients** are passed to the LLM, so explanations cite actual formulation.
2. **Strict prompt rules:** only listed products/ingredients, no invented links, no assumed concerns, professional tone.
3. **Safety boundary:** cosmetic guidance only — severe/medical-sounding concerns are redirected to a dermatologist.

*Residual known issue: a few haircare products leak the skincare filter (shared ingredients like glycerin/panthenol) — documented limitation, root cause in the Step 2 filter.*

In [0]:
# ============================================================
# CHATBOT — hybrid retrieve + grounded LLM response
# ============================================================
def ask_skincare_bot(query_text, n=5, show_products=False):
    """Full RAG chatbot: query -> detect concerns -> hybrid retrieve -> grounded response."""
    # 1. Detect concerns from free text
    detected = detect_concerns(query_text)

    # 2. Hybrid retrieve: semantic candidates, filtered by concern tags
    query_embedding = model.encode([query_text])[0].tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=30)
    matched = []
    for i in range(len(results["ids"][0])):
        pid = results["ids"][0][i]
        meta = results["metadatas"][0][i]
        pc = [c.strip() for c in meta["concerns"].split(",") if c.strip()]
        if any(c in pc for c in detected):
            matched.append((pid, meta))
        if len(matched) >= n:
            break

    # 3. Grounded product list WITH real ingredients
    products_text = ""
    for i, (pid, p) in enumerate(matched, 1):
        ings = ", ".join(id_to_ingredients.get(pid, [])[:8])
        products_text += (f"{i}. {p['brand']} (${p['price']:.0f})\n"
                          f"   Treats: {p['concerns']}\n"
                          f"   Ingredients: {ings}\n")

    # 4. Strict prompt (anti-hallucination + safety boundary)
    system_prompt = (
        "You are a knowledgeable, friendly skincare assistant. Follow these rules strictly:\n"
        "1. Recommend ONLY products from the provided list. Never invent products.\n"
        "2. When explaining why a product helps, refer ONLY to the ingredients and "
        "concerns actually listed for that product. Do NOT invent ingredients, "
        "benefits, claims, or product links.\n"
        "3. Address ONLY the concerns the user actually mentioned. Do NOT assume "
        "other concerns.\n"
        "4. Keep a warm but professional tone. No pet names.\n"
        "5. Give cosmetic guidance only. Do NOT diagnose medical conditions. If the "
        "concern sounds severe or medical, gently suggest seeing a dermatologist.")
    user_prompt = (
        f'User\'s message: "{query_text}"\n\n'
        f"The user's concern(s): {', '.join(detected)}\n\n"
        f"Products from our catalog (use ONLY these, and only their listed ingredients):\n"
        f"{products_text}\n"
        f"Write a warm, professional recommendation. Mention specific ingredients "
        f"from the list to explain why each product helps with the user's concern.")

    # 5. Generate
    response = client.chat.completions.create(
        model="databricks-meta-llama-3-1-8b-instruct",
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
        max_tokens=400)
    answer = response.choices[0].message.content

    if show_products:
        print(f"[Detected: {', '.join(detected)} | {len(matched)} products]\n")
    print(f"🧑 {query_text}\n")
    print(f"🤖 {answer}\n")
    print("-" * 60)
    return answer

print("Chatbot ready. Call: ask_skincare_bot('your question')")

Chatbot ready. Call: ask_skincare_bot('your question')


### Test — three different concerns

In [0]:
# Test the chatbot across distinct concerns
for q in ["my skin is really dry and flaky",
          "I have dark spots and uneven skin tone",
          "fine lines are starting to show and my skin feels less firm"]:
    ask_skincare_bot(q)

🧑 my skin is really dry and flaky

🤖 It sounds like your skin is struggling with dryness and flakiness. Don't worry, we have some great options to help you tackle this issue.

For dry and flaky skin, I'd recommend the This Works product. It contains glycerin and glyceryl stearate SE, which are both humectants that help to hydrate the skin and lock in moisture. This Works also includes candelilla/Jojoba/rice bran polyglyceryl-3 esters, which can help to soothe and calm dry skin. Give it a try!

If you're looking for a more intense treatment, the Dermalogica Super Rich Repair might be up your alley. It boasts a blend of nourishing ingredients like jojoba oil, shea butter, and hyaluronic-acid-free alternatives like dipropylene glycol and caprylic/capric/myristic/stearic triglyceride. These help to deeply moisturize your skin, reducing dryness and flakiness.

For a more affordable option, the R+co product contains butane, which can help to penetrate the skin and provide a quick burst of hy

## 5. Image path (the differentiator)

A user uploads a skin photo; **Llama 4 Maverick** (vision) analyzes it and returns a structured three-mode decision:

| Mode | Trigger | Action |
|---|---|---|
| `recommend` | normal cosmetic characteristics | detected concerns → chatbot recommendation |
| `escalate` | signs beyond cosmetic scope | dermatologist referral — **never names a condition** |
| `retake` | unusable photo | ask for a clearer image |

**Safety design:** the model *describes, never diagnoses*. Escalation is deliberately conservative — over-referring to a professional is the right failure mode for a medical-adjacent feature. All three branches were verified (recommend: stock face photo; retake: landscape photo; escalate: controlled logic test + a real photo that correctly escalated).

In [0]:
# ============================================================
# IMAGE PATH — three-mode vision analysis
# ============================================================
import base64, requests, json

VISION_PROMPT = """You are a cosmetic skin-image assistant. Look at the photo and decide ONE of:
1. 'recommend' — you see normal COSMETIC characteristics (oiliness, dryness, redness, dark spots, dullness, texture, large pores, fine lines).
2. 'escalate' — you see signs BEYOND cosmetic scope (significant inflammation, lesions, wounds, or anything unusual/concerning). Do NOT name any condition.
3. 'retake' — the photo is unusable (blurry, too dark, or no face/skin visible).

Reply ONLY with JSON in this exact format, nothing else:
{"decision": "recommend or escalate or retake", "concerns": ["list", "of", "cosmetic", "concern", "words"], "observation": "one short sentence describing what you see"}

Remember: describe only, never diagnose a medical condition."""

def analyze_skin_image(image_url=None, image_bytes=None):
    """Vision model returns a structured decision: recommend / escalate / retake."""
    if image_url:
        resp = requests.get(image_url, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        image_bytes = resp.content
    image_data = base64.b64encode(image_bytes).decode("utf-8")

    completion = client.chat.completions.create(
        model="databricks-llama-4-maverick",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": VISION_PROMPT},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}}]
        }],
        max_tokens=300)

    raw = completion.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except Exception:
        # Safe default: never guess on an unparseable analysis
        return {"decision": "retake", "concerns": [],
                "observation": "Could not analyze the image clearly."}


def handle_skin_image(image_url=None, image_bytes=None):
    """Route the vision decision: recommend -> chatbot, escalate -> referral, retake -> re-ask."""
    result = analyze_skin_image(image_url=image_url, image_bytes=image_bytes)
    decision = result.get("decision")
    concerns = result.get("concerns", [])
    observation = result.get("observation", "")

    print(f"📷 Image analysis: {observation}\n")

    if decision == "escalate":
        print("🩺 Based on what's visible in your photo, I'd recommend having this "
              "looked at by a dermatologist for a proper evaluation. I'm only able "
              "to offer cosmetic skincare guidance, so a professional is the right "
              "next step here.")
        return
    if decision == "retake":
        print("📸 I couldn't analyze your skin clearly from this photo. Could you "
              "share a clearer, well-lit photo of your face?")
        return
    if decision == "recommend":
        if not concerns:
            print("I couldn't detect a specific concern. Could you tell me what "
                  "you'd like help with?")
            return
        print(f"✨ I can see signs of: {', '.join(concerns)}. "
              f"Here are some products that may help:\n")
        ask_skincare_bot(f"my skin shows {' and '.join(concerns)}")
        return
    print("Something went wrong analyzing the image. Please try again.")

print("Image path ready. Call: handle_skin_image(image_url=...) or (image_bytes=...)")

Image path ready. Call: handle_skin_image(image_url=...) or (image_bytes=...)


### Test — image path
To test with your own photo, upload it to the volume and set the path below. (Use only photos you have the right to use.)

In [0]:
# Test with an image from the volume (upload one first, adjust path)
my_image_path = "/Volumes/dermstore/dermstoreschema/dermstorevolume/myfiles/test_photo.jpg"

with open(my_image_path, "rb") as f:
    image_bytes = f.read()
handle_skin_image(image_bytes=image_bytes)

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
File <command-7515342453286123>, line 4
      1 # Test with an image from the volume (upload one first, adjust path)
      2 my_image_path = "/Volumes/dermstore/dermstoreschema/dermstorevolume/myfiles/test_photo.jpg"
----> 4 with open(my_image_path, "rb") as f:
      5     image_bytes = f.read()
      6 handle_skin_image(image_bytes=image_bytes)

File /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:324, in _modified_open(file, *args, **kwargs)
    317 if file in {0, 1, 2}:
    318     raise ValueError(
    319         f"IPython won't let you open fd={file} by default "
    320         "as it is likely to crash IPython. If you know what you are doing, "
    321         "you can use builtins' open."
    322     )
--> 324 return io_open(file, *args, **kwargs)

FileNotFoundError: [Errno 2] No such file or di

## 6. Known limitations & next steps

**Measured & documented (see Step 4 evaluation notebook):**
- Concern-tagging spot-check on 18 ground-truth products: recall 0.28 / precision 0.10 (v2 tags). A stricter re-tune (v3) *worsened* both metrics and was reverted — full experiment history in MLflow (`skincare_concern_tagging`).
- A few haircare products pass the ingredient-evidence skincare filter via shared ingredients — a category-signal + ingredient-evidence hybrid filter is the planned fix.
- Ground truth is sparse (18 products) — metrics are directional, not definitive.

**Production path:** Databricks Vector Search (Delta-synced index) instead of in-memory Chroma; larger labeled evaluation set; dermatologist-informed escalation threshold; live weather/location features.

In [0]:
# ============================================================
# APP PREP — Export product data as JSON for the app to bundle
# ============================================================
import json

records = []
for _, row in pdf.iterrows():
    records.append({
        "product_id": str(row["product_id"]),
        "brand": str(row["brand"]),
        "price": float(row["price"]) if row["price"] is not None else 0.0,
        "concerns": list(row["concerns"]) if row["concerns"] is not None else [],
        "ingredients": list(row["ingredients_list"][:8]) if row["ingredients_list"] is not None else [],
        "document": row["document"],          # <- renamed from document_v2
    })

out_path = "/Volumes/dermstore/dermstoreschema/dermstorevolume/myfiles/app_data.json"
with open(out_path, "w") as f:
    json.dump(records, f)

print(f"Exported {len(records)} products to {out_path}")

Exported 106 products to /Volumes/dermstore/dermstoreschema/dermstorevolume/myfiles/app_data.json
